# TikTok strategy — five-day loser mean reversion

The video describes a contrarian rule: buy stocks that fell during the previous week and sell stocks that rose. The Week 1 simulator is intentionally **long-only**, so this notebook implements the compatible version: buy recent losers in proportion to their decline, assign recent winners zero weight, and hold cash when no stock declined.

At day `t`, the strategy uses only the five returns ending at `t`; those weights earn the return from `t` to `t+1`.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.observation import build_observation
from tradinglab.strategies.mean_reversion import weekly_loser_weights

feed = DataFeed.from_dir('data/egx')
print('assets:', feed.n_assets, '| days:', feed.n_days)

## Recreate the strategy once, visibly

Feature 0 of an observation contains daily returns. Compounding the latest five returns produces the previous trading week's return. A negative period return becomes positive buying strength; positive period returns are excluded because short selling is outside this simulator's contract.

In [ ]:
day = 500
lookback = 30
obs = build_observation(feed, day, lookback)

five_day_returns = np.prod(1 + obs[:, -5:, 0], axis=1) - 1
weights = weekly_loser_weights(obs, lookback_days=5)

print(f'decision date: {feed.dates[day]}')
print(f'{"asset":8s} {"5-day return":>14s} {"weight":>10s} {"EGP of 1000":>13s}')
for symbol, period_return, weight in zip(feed.symbols, five_day_returns, weights):
    print(f'{symbol:8s} {period_return:14.2%} {weight:10.2%} {1000 * weight:13.2f}')

print('weight sum:', weights.sum())

## Sanity checks

The strategy must be long-only, must never invest more than 100%, and must allocate more weight to a larger decline.

In [ ]:
toy = np.zeros((3, 5, 5))
toy[0, :, 0] = -0.01
toy[1, :, 0] = 0.01
toy[2, :, 0] = -0.02
toy_weights = weekly_loser_weights(toy, lookback_days=5)

assert np.all(toy_weights >= 0)
assert np.isclose(toy_weights.sum(), 1.0)
assert toy_weights[1] == 0
assert toy_weights[2] > toy_weights[0]
print('strategy checks passed ✓', toy_weights.round(3))

## Next: the complete simulator

Notebook 4 runs `weekly_loser_weights` through the same `PortfolioSimulator` used by the SMA strategy, with the same 1,000 EGP starting value, commission setting, dates, benchmark, and reporting functions. That makes the comparison fair.